In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import os
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [3]:
AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
N_AA = len(AA_LIST)
aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
idx_to_aa = {v: k for k, v in aa_to_idx.items()}

def min_max_norm(data):
    data_min = np.min(data)
    data_max = np.max(data)
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def magnitude_norm(data, data_min=0):
    data_max = np.max(np.abs(data))
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def get_look_at_edges(G, source_node, sort=True):
    all_edges = []
    for neighbor, edge_data in G[source_node].items():
        weight = edge_data.get('weight')
        all_edges.append((source_node, neighbor, weight))
    if sort:
        return sorted(all_edges, key=lambda x: x[2], reverse=True)
    else:
        return all_edges

In [4]:
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
attr_dir = "results/attribution_maps"
attr_class = "class_1"
attn_dir = "results/attention_maps"
data_dir = "data"
task_type = "classification"
data_df = pd.read_csv(os.path.join(root_dir, data_dir, "input_info_VRC01_IC80.csv"))
aligned_sequences = np.array([list(seq) for seq in data_df['Sequence']])
seq_no_array = data_df['Seq_no'].values
class_label = data_df["Label"].values
resno_df = pd.read_csv(os.path.join(root_dir, data_dir, "selected_residues_IC80.csv"))
resno_array = resno_df["ResLabel"].values

In [5]:
N_MODEL = len(selected_models)
N_SEQ, L_SEQ = aligned_sequences.shape

attr_scalar_array = np.zeros((N_MODEL, N_SEQ, L_SEQ))
attn_scalar_array = np.zeros((N_MODEL, N_SEQ, L_SEQ, L_SEQ))
cls_attn_scalar_array =  np.zeros([N_MODEL, N_SEQ, L_SEQ])

for n, model_name in enumerate(selected_models):
    for i, no in enumerate(data_df['Seq_no']):
        attr = np.load(os.path.join(root_dir, attr_dir, task_type, model_name, str(no)+"_attribution_ig_"+attr_class+".npy"))
        attr_scalar_array[n, i] = magnitude_norm(attr)

        attn = np.load(os.path.join(root_dir, attn_dir, task_type, "full", model_name, str(no)+"_attentions.npy"))
        attn_arr = attn[1:, 1:]
        attn_scalar_array[n, i] = min_max_norm(attn_arr)
        cls_attn = attn[0, 1:]
        cls_attn_scalar_array[n,i] = min_max_norm(cls_attn)

mean_attr_scalar_array = np.mean(attr_scalar_array, axis=0)
mean_attn_scalar_array = np.mean(attn_scalar_array, axis=0)
mean_cls_attn_scalar_array = np.mean(cls_attn_scalar_array, axis=0)

In [6]:
attr_j_broadcastable = mean_attr_scalar_array[:, np.newaxis, :]
weighted_contributions = mean_attn_scalar_array * attr_j_broadcastable

sort_indices = np.argsort(-np.abs(weighted_contributions), axis=2)
sorted_contributions = np.take_along_axis(weighted_contributions, sort_indices, axis=2)
trajectories = np.cumsum(sorted_contributions, axis=2)
sorted_resno = resno_array[sort_indices]

# trajectories_raw = np.cumsum(sorted_contributions, axis=2)
# trajectories_with_zero = np.pad(
#     trajectories_raw,
#     ((0, 0), (0, 0), (1, 0)),
#     'constant',
#     constant_values=0
# )

# start_points = mean_attr_scalar_array[:, :, np.newaxis]
# trajectories_long = trajectories_with_zero + start_points
# trajectories = trajectories_long[:, :, :-1]

In [7]:
# result_dir = "results/graphs"
# attr_class = "class_1"
# weighted_strategy = ["Source"]
# contributions = ["Negative", "Positive"]

# graph_collection = {contribution:[] for contribution in contributions}
# for contribution in contributions:
#     graph_collection[contribution] = {weighted_by:[] for weighted_by in weighted_strategy}
#     for weighted_by in weighted_strategy:
#         graphfile = os.path.join(root_dir, result_dir, f"Graph_{attr_class}_label_all_{weighted_by}_{contribution}.gml")
#         G = nx.read_gml(graphfile)
#         graph_collection[contribution][weighted_by] = G

In [125]:
k = 50
n_select = 3
steps = 10
focus_residues = [(279, 'D'), (460, 'S'), (281, 'A'), (281, 'T'), (276, 'N')]

for mut_resno, mut_aa in focus_residues:
    resno_pos = np.where(resno_array == str(mut_resno))[0]
    mut_mask = aligned_sequences[:, resno_pos].reshape(-1) == mut_aa
    mut_seq_no = seq_no_array[mut_mask]
    mut_label = data_df["Label"].values[mut_mask]
    mut_seq = aligned_sequences[mut_mask]

    selected_attr = mean_attr_scalar_array[mut_mask]
    top_k_indices = np.argsort(np.abs(selected_attr), axis=1)[:, -k:]
    top_k_indices_high_to_low = top_k_indices[:, ::-1]
    top_k_values = np.take_along_axis(selected_attr, top_k_indices_high_to_low, axis=1)
    sum_of_top_k_attr = top_k_values.sum(axis=1)

    fitness_from_attr = []
    for sum_attr in sum_of_top_k_attr:
        if sum_attr < 0:
            fit = 0
        else:
            fit = 1
        fitness_from_attr.append(fit)
    
    top_k_resno_array = resno_array[top_k_indices_high_to_low]
    row_indices, rank_indices = np.where((top_k_resno_array == str(mut_resno)))
    rank_of_mut_resno = np.full(top_k_resno_array.shape[0], k + 1)
    rank_of_mut_resno[row_indices] = rank_indices + 1

    mut_feature_index = resno_pos[0]
    abs_mut_attr = np.abs(selected_attr[:, mut_feature_index])
    abs_all_attr = np.abs(selected_attr)
    num_features_le = (abs_all_attr <= abs_mut_attr[:, np.newaxis]).sum(axis=1)
    mut_percentile = (num_features_le / selected_attr.shape[1]) * 100

    candidate_mask = (top_k_resno_array == str(mut_resno)).any(axis=1) #& (fitness_from_attr == mut_label)
    mut_attr_score = top_k_values[candidate_mask][resno_array[top_k_indices_high_to_low[candidate_mask]] == str(mut_resno)]
    
    # print(f"Analyzing mutation {mut_resno}{mut_aa}. True label counts: {np.unique(mut_label, return_counts=True)}")
    # print(f"Accuracy: {accuracy_score(fitness_from_attr, mut_label):.4f} | Correct predictions: {(fitness_from_attr == mut_label).sum()} / Total samples: {(mut_mask).sum()}")
    # print(f"Correctly predicted true labels (counts): {np.unique(mut_label[fitness_from_attr == mut_label], return_counts=True)}")
    # print(f"Count of correct predictions where mutation residue is top {k} feature: {candidate_mask.sum()}")
    # print(f"Breakdown of correct predictions (0/1) when residue is top {k}: {np.unique(mut_label[candidate_mask], return_counts=True)}")
    # print(f"Mutation residue's attribution score (Positive/Negative counts): {(mut_attr_score > 0).sum()} / {(mut_attr_score < 0).sum()}")
    # print("------")
    
    mask_pos = mut_attr_score > 0
    pos_aa_attr = mut_attr_score[mask_pos]
    pos_seq = mut_seq[candidate_mask][mask_pos]
    pos_label = mut_label[candidate_mask][mask_pos]
    pos_seq_no = mut_seq_no[candidate_mask][mask_pos]
    pos_rank = rank_of_mut_resno[candidate_mask][mask_pos]
    pos_percentile = mut_percentile[candidate_mask][mask_pos]
    pos_trajectories = trajectories[mut_mask][candidate_mask][mask_pos]
    pos_sorted_idx = sort_indices[mut_mask][candidate_mask][mask_pos]
    if len(pos_aa_attr) > 1:
        pos_aa_attr = pos_aa_attr[np.argsort(-pos_percentile)[:n_select]]
        pos_seq = pos_seq[np.argsort(-pos_percentile)[:n_select]]
        pos_label = pos_label[np.argsort(-pos_percentile)[:n_select]]
        pos_seq_no = pos_seq_no[np.argsort(-pos_percentile)[:n_select]]
        pos_rank = pos_rank[np.argsort(-pos_percentile)[:n_select]]
        pos_percentile = pos_percentile[np.argsort(-pos_percentile)[:n_select]]
        pos_trajectories = pos_trajectories[np.argsort(-pos_percentile)[:n_select]]
        pos_sorted_idx = pos_sorted_idx[np.argsort(-pos_percentile)[:n_select]]
    
    mask_neg = mut_attr_score < 0
    neg_aa_attr = mut_attr_score[mask_neg]
    neg_seq = mut_seq[candidate_mask][mask_neg]
    neg_label = mut_label[candidate_mask][mask_neg]
    neg_seq_no = mut_seq_no[candidate_mask][mask_neg]
    neg_rank = rank_of_mut_resno[candidate_mask][mask_neg]
    neg_percentile = mut_percentile[candidate_mask][mask_neg]
    neg_trajectories = trajectories[mut_mask][candidate_mask][mask_neg]
    neg_sorted_idx = sort_indices[mut_mask][candidate_mask][mask_neg]
    if len(neg_aa_attr) > 1:
        neg_aa_attr = neg_aa_attr[np.argsort(-neg_percentile)[:n_select]]
        neg_seq = neg_seq[np.argsort(-neg_percentile)[:n_select]]
        neg_label = neg_label[np.argsort(-neg_percentile)[:n_select]]
        neg_seq_no = neg_seq_no[np.argsort(-neg_percentile)[:n_select]]
        neg_rank = neg_rank[np.argsort(-neg_percentile)[:n_select]]
        neg_percentile = neg_percentile[np.argsort(-neg_percentile)[:n_select]]
        neg_trajectories = neg_trajectories[np.argsort(-neg_percentile)[:n_select]]
        neg_sorted_idx = neg_sorted_idx[np.argsort(-neg_percentile)[:n_select]]
    
    all_attr_score = np.concatenate([pos_aa_attr, neg_aa_attr])
    all_seq = np.concatenate([pos_seq, neg_seq])
    all_label = np.concatenate([pos_label, neg_label])
    all_seq_no = np.concatenate([pos_seq_no, neg_seq_no])
    all_rank = np.concatenate([pos_rank, neg_rank])
    all_percentile = np.concatenate([pos_percentile, neg_percentile])
    all_trajectories  = np.concatenate([pos_trajectories, neg_trajectories])
    all_sorted_idx = np.concatenate([pos_sorted_idx, neg_sorted_idx])
    
    mut_all_trajectories = all_trajectories[:, resno_pos, :steps]
    mut_all_sorted_idx = all_sorted_idx[:, resno_pos, :steps]
    mut_all_aa_trajectories = resno_array[mut_all_sorted_idx]
    mut_all_seq = np.array([all_seq[i, mut_all_sorted_idx[i][0]] for i in range(len(mut_all_sorted_idx))])

    print(f"--- Displaying Trajectories for {mut_resno}_{mut_aa} Sequences ---")
    for s, (seq, label, seq_no) in enumerate(zip(all_seq, all_label, all_seq_no)):
        print(f"\n=== Sequence: {seq_no}, Label: {label}, Attr score: {all_attr_score[s]} ===")
        print(f"Step | Attended ResNo | AA | Cumulative Score")
        print("---------------------------------------------")
        for st in range(steps):
            resno = mut_all_aa_trajectories[s, 0, st]
            aa = mut_all_seq[s, st]
            score = mut_all_trajectories[s, 0, st]
            print(f" {st:<4} | {resno:<14} | {aa:<2} | {score:+.4f}")
        print("---------------------------------------------")

--- Displaying Trajectories for 279_D Sequences ---

=== Sequence: 1961, Label: 1, Attr score: 0.297672963142395 ===
Step | Attended ResNo | AA | Cumulative Score
---------------------------------------------
 0    | 58             | A  | +0.3277
 1    | 84             | M  | +0.6271
 2    | 46             | K  | +0.8228
 3    | 47             | E  | +0.9591
 4    | 236            | K  | +0.8229
 5    | 643            | Y  | +0.7227
 6    | 165            | I  | +0.8223
 7    | 44             | V  | +0.9101
 8    | 277            | L  | +0.9900
 9    | 283            | T  | +0.9131
---------------------------------------------

=== Sequence: 359, Label: 1, Attr score: 0.3838586416095495 ===
Step | Attended ResNo | AA | Cumulative Score
---------------------------------------------
 0    | 84             | I  | +0.3760
 1    | 46             | K  | +0.5772
 2    | 47             | E  | +0.7278
 3    | 638            | Y  | +0.8601
 4    | 58             | A  | +0.9809
 5    | 44        

In [239]:
# k = 50
# n_select = 5
# steps = 20
# focus_residues = [(279, 'D'), (460, 'S'), (281, 'A'), (281, 'T'), (276, 'N')]

# for mut_resno, mut_aa in focus_residues:
#     resno_pos = np.where(resno_array == str(mut_resno))[0]
#     mut_mask = aligned_sequences[:, resno_pos].reshape(-1) == mut_aa
#     mut_seq_no = seq_no_array[mut_mask]
#     mut_label = data_df["Label"].values[mut_mask]
#     mut_seq = aligned_sequences[mut_mask]

#     selected_attr = mean_attr_scalar_array[mut_mask]
#     top_k_indices = np.argsort(np.abs(selected_attr), axis=1)[:, -k:]
#     top_k_indices_high_to_low = top_k_indices[:, ::-1]
#     top_k_values = np.take_along_axis(selected_attr, top_k_indices_high_to_low, axis=1)
#     sum_of_top_k_attr = top_k_values.sum(axis=1)

#     fitness_from_attr = []
#     for sum_attr in sum_of_top_k_attr:
#         if sum_attr < 0:
#             fit = 0
#         else:
#             fit = 1
#         fitness_from_attr.append(fit)
    
#     top_k_resno_array = resno_array[top_k_indices_high_to_low]
#     row_indices, rank_indices = np.where((top_k_resno_array == str(mut_resno)))
#     rank_of_mut_resno = np.full(top_k_resno_array.shape[0], k + 1)
#     rank_of_mut_resno[row_indices] = rank_indices + 1

#     mut_feature_index = resno_pos[0]
#     abs_mut_attr = np.abs(selected_attr[:, mut_feature_index])
#     abs_all_attr = np.abs(selected_attr)
#     num_features_le = (abs_all_attr <= abs_mut_attr[:, np.newaxis]).sum(axis=1)
#     mut_percentile = (num_features_le / selected_attr.shape[1]) * 100
    
#     print(f"Analyzing mutation {mut_resno}{mut_aa}. True label counts: {np.unique(mut_label, return_counts=True)}")
#     print(f"Accuracy: {accuracy_score(fitness_from_attr, mut_label):.4f} | Correct predictions: {(fitness_from_attr == mut_label).sum()} / Total samples: {(mut_mask).sum()}")
#     print(f"Correctly predicted true labels (counts): {np.unique(mut_label[fitness_from_attr == mut_label], return_counts=True)}")
    
#     candidate_mask = (top_k_resno_array == str(mut_resno)).any(axis=1) & (fitness_from_attr == mut_label)
#     print(f"Count of correct predictions where mutation residue is top {k} feature: {candidate_mask.sum()}")
#     print(f"Breakdown of correct predictions (0/1) when residue is top {k}: {np.unique(mut_label[candidate_mask], return_counts=True)}")
#     mut_attr_score = top_k_values[candidate_mask][resno_array[top_k_indices_high_to_low[candidate_mask]] == str(mut_resno)]
#     print(f"Mutation residue's attribution score (Positive/Negative counts): {(mut_attr_score > 0).sum()} / {(mut_attr_score < 0).sum()}")
#     print("------")
    
#     mask_pos = mut_attr_score > 0
#     pos_aa_attr = mut_attr_score[mask_pos]
#     pos_seq = mut_seq[candidate_mask][mask_pos]
#     pos_label = mut_label[candidate_mask][mask_pos]
#     pos_seq_no = mut_seq_no[candidate_mask][mask_pos]
#     pos_rank = rank_of_mut_resno[candidate_mask][mask_pos]
#     pos_percentile = mut_percentile[candidate_mask][mask_pos]
#     if len(pos_aa_attr) > 1:
#         pos_aa_attr = pos_aa_attr[np.argsort(-pos_percentile)[:n_select]]
#         pos_seq = pos_seq[np.argsort(-pos_percentile)[:n_select]]
#         pos_label = pos_label[np.argsort(-pos_percentile)[:n_select]]
#         pos_seq_no = pos_seq_no[np.argsort(-pos_percentile)[:n_select]]
#         pos_rank = pos_rank[np.argsort(-pos_percentile)[:n_select]]
#         pos_percentile = pos_percentile[np.argsort(-pos_percentile)[:n_select]]
    
#     mask_neg = mut_attr_score < 0
#     neg_aa_attr = mut_attr_score[mask_neg]
#     neg_seq = mut_seq[candidate_mask][mask_neg]
#     neg_label = mut_label[candidate_mask][mask_neg]
#     neg_seq_no = mut_seq_no[candidate_mask][mask_neg]
#     neg_rank = rank_of_mut_resno[candidate_mask][mask_neg]
#     neg_percentile = mut_percentile[candidate_mask][mask_neg]
#     if len(neg_aa_attr) > 1:
#         neg_aa_attr = neg_aa_attr[np.argsort(-neg_percentile)[:n_select]]
#         neg_seq = neg_seq[np.argsort(-neg_percentile)[:n_select]]
#         neg_label = neg_label[np.argsort(-neg_percentile)[:n_select]]
#         neg_seq_no = neg_seq_no[np.argsort(-neg_percentile)[:n_select]]
#         neg_rank = neg_rank[np.argsort(-neg_percentile)[:n_select]]
#         neg_percentile = neg_percentile[np.argsort(-neg_percentile)[:n_select]]
    
#     all_attr_score = np.concatenate([pos_aa_attr, neg_aa_attr])
#     all_seq = np.concatenate([pos_seq, neg_seq])
#     all_label = np.concatenate([pos_label, neg_label])
#     all_seq_no = np.concatenate([pos_seq_no, neg_seq_no])
#     all_rank = np.concatenate([pos_rank, neg_rank])
#     all_percentile = np.concatenate([pos_percentile, neg_percentile])

#     start_node = str(mut_resno)+"_"+mut_aa
#     ranked_edges_positive = get_look_at_edges(graph_collection['Positive']['Source'], source_node)
#     ranked_edges_negative = get_look_at_edges(graph_collection['Negative']['Source'], source_node)

#     start_pos = [_[2] for _ in ranked_edges_positive if _[1] == start_node]
#     start_pos = start_pos[0] if len(start_pos) == 1 else 0

#     start_neg = [_[2] for _ in ranked_edges_negative if _[1] == start_node]
#     start_neg = start_neg[0] if len(start_neg) == 1 else 0

#     start_point = start_pos - start_neg

#     step_dict = {"ResLabel":[], "AA":[], "Step":[]}
#     for neg_list, pos_list in zip(ranked_edges_negative, ranked_edges_positive):
#         reslabel, AA = neg_list[1].split("_")
#         step_dict["ResLabel"].append(reslabel)
#         step_dict["AA"].append(AA)
#         step_dict["Step"].append(-neg_list[2])

#         reslabel, AA = pos_list[1].split("_")
#         step_dict["ResLabel"].append(reslabel)
#         step_dict["AA"].append(AA)
#         step_dict["Step"].append(pos_list[2])
#     step_df = pd.DataFrame(step_dict)
#     step_df["abs_score"] = np.abs(step_df["Step"])
#     step_df.sort_values(by=["abs_score"], ascending=False, inplace=True)

#     resno_mask = np.isin(resno_array, np.unique(step_df["ResLabel"]))
#     all_reslabel = resno_array[resno_mask]

#     #fitness = []
#     trajectories = np.zeros([len(all_seq), steps])
#     aa_trajectories = np.empty([len(all_seq), steps], dtype=object)
#     seq_trajectories = np.empty(len(all_seq), dtype=object)
#     seq_no_trajectories = np.empty(len(all_seq), dtype=object)
#     for s, (seq, label, seq_no) in enumerate(zip(all_seq, all_label, all_seq_no)):
#         route = start_point
#         sub_seq = seq[resno_mask]
#         seq_trajectories[s] = sub_seq
#         seq_no_trajectories[s] = seq_no
#         seq_lookup = dict(zip(all_reslabel, sub_seq))
#         count_k = 0
#         for row in step_df.itertuples():
#             if seq_lookup.get(row.ResLabel) == row.AA:
#                 route += row.Step
#                 trajectories[s, count_k] = row.Step
#                 aa_trajectories[s, count_k] = row.ResLabel+row.AA
#                 count_k += 1
#             if count_k == steps:
#                 break
#         # if route < 0:
#         #     fit = 0
#         # else:
#         #     fit = 1
#         # fitness.append(fit)
#     print(all_label)
#     #print(fitness)
#     print(all_rank)
#     print(all_percentile)
#     print(trajectories.sum(axis=1))
#     print(all_attr_score)
#     print("___________________________")

Analyzing mutation 279D. True label counts: (array([0, 1]), array([301, 153]))
Accuracy: 0.9339 | Correct predictions: 424 / Total samples: 454
Correctly predicted true labels (counts): (array([0, 1]), array([291, 133]))
Count of correct predictions where mutation residue is top 50 feature: 187
Breakdown of correct predictions (0/1) when residue is top 50: (array([0, 1]), array([88, 99]))
Mutation residue's attribution score (Positive/Negative counts): 186 / 1
------
[1 1 1 1 0 0]
[ 6  6  6  8  8 28]
[97.6076555  97.6076555  97.6076555  96.6507177  96.6507177  87.08133971]
[0.02741589 0.02088388 0.00385513 0.04250468 0.030505   0.04950563]
[ 0.38385864  0.31612294  0.29767296  0.27143865  0.24864701 -0.13460499]
___________________________
Analyzing mutation 460S. True label counts: (array([0, 1]), array([48, 26]))
Accuracy: 0.9730 | Correct predictions: 72 / Total samples: 74
Correctly predicted true labels (counts): (array([0, 1]), array([46, 26]))
Count of correct predictions where 